<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">


# Python for Finance, 3rd Edition
## Chapter 06 · Data Analysis with pandas

&copy; Dr. Yves J. Hilpisch<br>
AI-supported by GPT 5.x<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook mirrors the chapter examples in a Colab-ready format so that you
can run, tweak, and extend them interactively.


### How to Use This Notebook
- Run the cells top to bottom the first time so the objects or plots are
  created in order.
- Add your own cells for experiments or refactorings.
- Use the book text for the surrounding explanation and context.


`pandas` builds on `NumPy` to provide labeled, table-like data structures that
make real-world financial datasets easier to explore, clean, and analyze. This
chapter introduces `Series` and `DataFrame` objects, shows how to work with
time-indexed data, and prepares you for later chapters on time series, I/O,
and asset management.


# Why pandas for Finance


`pandas` is the right tool when you need labels, alignment, and time-aware
indexing.


# Core Data Structures: Series and DataFrame


`Series` handles one-dimensional labeled data, and `DataFrame` stores aligned
columns.


## Creating a Series


A `Series` is the simplest labeled container for prices, returns, or any
single field.


In [ ]:
import pandas as pd

In [ ]:
import numpy as np

In [ ]:
prices = pd.Series(
    [100.0, 101.5, 103.2],
    index=["2026-01-02", "2026-01-05", "2026-01-06"],
    name="price",
)  # Create a labeled vector of prices with explicit date-like index labels.

In [ ]:
prices

## Creating a DataFrame


`DataFrame` objects combine multiple aligned series into a table.


In [ ]:
data = {
    "price": [100.0, 101.5, 103.2],
    "volume": [1_000, 1_500, 800],
}

In [ ]:
# Construct a `DataFrame` with price and volume columns sharing a date-like
# index.
df = pd.DataFrame(data, index=["2026-01-02", "2026-01-05", "2026-01-06"])

In [ ]:
df

In [ ]:
# The index labels the rows; here they are strings that happen to look like
# dates.
df.index

In [ ]:
df.columns  # The columns label each series in the table.

In [ ]:
df.dtypes  # `dtypes` shows the underlying `NumPy` dtypes used for each column.

# Indexing and Selection


Label-aware selection is one of the main reasons to use `pandas` instead of
raw arrays.


## Column and Row Selection


Selecting a column returns a `Series`, while selecting multiple columns keeps
a `DataFrame`.


In [ ]:
prices = df["price"]  # Selecting a single column returns a `Series`.

In [ ]:
prices

In [ ]:
# Selecting a list of columns returns a `DataFrame` with those columns only.
sub = df[["price", "volume"]]

In [ ]:
sub

## Label-Based Selection with .loc[]


`loc` matches labels, which makes row and column selection explicit.


In [ ]:
df_loc = df.loc["2026-01-05"]  # Select a single row by its index label.

In [ ]:
df_loc

In [ ]:
# Select a label-based slice of rows and a subset of columns.
df_slice = df.loc["2026-01-02":"2026-01-05", ["price"]]

In [ ]:
df_slice

## Position-Based Selection with .iloc[]


`iloc` works with integer positions when labels are not the right tool.


In [ ]:
first_row = df.iloc[0]  # Select the first row by position.

In [ ]:
first_row

In [ ]:
# Select the first two rows of the first column by position.
head_two = df.iloc[:2, 0]


In [ ]:
head_two

# Working with Time-Indexed Data


Datetime indexes let `pandas` track market data at daily or intraday
frequency.


## Using DatetimeIndex


A `DatetimeIndex` makes date arithmetic, alignment, and resampling
straightforward.


In [ ]:
df_ts = df.copy()

In [ ]:
# Convert the string index into a `DatetimeIndex` so `pandas` can treat it as
# calendar data.
df_ts.index = pd.to_datetime(df_ts.index)

In [ ]:
df_ts

In [ ]:
# Partial string indexing selects all rows in January 2026.
df_ts.loc["2026-01"]


## Resampling and Aggregation


Resampling turns daily observations into weekly or monthly summaries.


In [ ]:
# Resample daily prices to weekly, taking the last observed price in each
# week.
weekly = df_ts["price"].resample("W").last()

In [ ]:
weekly

# Handling Missing Data


Real datasets contain gaps, and `pandas` gives you direct tools for finding
and filling them.


## Detecting Missing Values


Missing-value checks show exactly where the data is incomplete.


In [ ]:
df_na = df_ts.copy()

In [ ]:
# Introduce a missing price for demonstration purposes.
df_na.loc["2026-01-05", "price"] = np.nan


In [ ]:
df_na

In [ ]:
# `.isna()` returns a Boolean `DataFrame` indicating missing entries.
df_na.isna()


## Dropping or Filling Missing Values


You can either remove incomplete rows or replace them with a sensible
estimate.


In [ ]:
# Drop rows where the `price` column is missing.
df_drop = df_na.dropna(subset=["price"])


In [ ]:
df_drop

In [ ]:
# Forward-fill missing values, propagating the last known price forward in
# time.
df_ffill = df_na.ffill()

In [ ]:
df_ffill

# Grouped Operations and Simple Aggregations


Grouping collapses repeated categories into compact summaries.


## Grouping by Symbol


Grouping by symbol is the basic pattern for handling multiple assets at once.


In [ ]:
data = {
    "date": ["2026-01-02", "2026-01-02", "2026-01-05", "2026-01-05"],
    "symbol": ["AAPL", "MSFT", "AAPL", "MSFT"],
    "price": [180.0, 350.0, 182.0, 355.0],
}

In [ ]:
# Create a simple multi-symbol price table in "long" format.
quotes = pd.DataFrame(data)


In [ ]:
quotes

In [ ]:
# Group by symbol and compute the mean of the `price` column in each group.
mean_price = quotes.groupby("symbol")["price"].mean()

In [ ]:
mean_price

## Multiple Aggregations


Multiple aggregations let you summarize each group with more than one
statistic.


In [ ]:
# Compute mean, minimum, and maximum price per symbol in one grouped
# operation.
stats = quotes.groupby("symbol")["price"].agg(["mean", "min", "max"])

In [ ]:
stats

# Computing Simple Returns


Simple returns measure percentage change from one price to the next.


In [ ]:
series = pd.Series(
    [100.0, None, 102.0, None],
    index=pd.to_datetime([
        "2026-01-02",
        "2026-01-05",
        "2026-01-06",
        "2026-01-07",
    ]),
)


In [ ]:
# `.pct_change()` computes period-over-period percentage returns.
rets = prices.pct_change()


In [ ]:
rets

In [ ]:
# Quick assert that confirms `.pct_change()` behaves as expected for a single
# step before you trust it on longer series.
assert rets.loc["2026-01-05"] == (101.5 / 100.0) - 1

# Reading Data from CSV Files


CSV input is the common bridge between raw market files and `pandas` tables.


In [ ]:
from io import StringIO

In [ ]:
csv_text = """date,symbol,price,volume
2026-01-02,AAPL,180.0,1000
2026-01-05,AAPL,182.0,1500
"""


In [ ]:
csv_file = StringIO(csv_text)

In [ ]:
df_csv = pd.read_csv(
    csv_file,
    parse_dates=["date"],
    index_col="date",
# Read the CSV while parsing the `date` column as datetimes and using it as
# the index.
)

In [ ]:
# Inspect dtypes to confirm that numeric columns are parsed as numbers and
# dates as datetimes.
df_csv.dtypes

In [ ]:
# Use `.head()` to check that the resulting table matches your expectations
# before moving on.
df_csv.head()

In [ ]:
import pandas as pd

In [ ]:
try:
    df_real = pd.read_csv(
        "../data/prices.csv",
        parse_dates=["date"],
        index_col="date",
    )
except FileNotFoundError as exc:
    raise SystemExit(f"Input file not found: {exc.filename}")
except pd.errors.ParserError as exc:
    raise SystemExit(f"CSV parsing failed: {exc}")


# Interoperability with NumPy


`pandas` and `NumPy` interoperate cleanly when you need labels in one step and
arrays in the next.


In [ ]:
# Extract the underlying `NumPy` array from a `Series`.
values = df_ts["price"].to_numpy()


In [ ]:
values

In [ ]:
arr = np.array([100.0, 101.5, 103.2])

In [ ]:
# Wrap a `NumPy` array into a `Series` with the same index.
s = pd.Series(arr, index=df_ts.index, name="price")

In [ ]:
s

In [ ]:
import numpy.typing as npt

In [ ]:
# Explicitly state that the function expects a `Series` and returns a
# floating-point `ndarray`, which matches how you use it elsewhere.
def to_price_array(series: pd.Series) -> npt.NDArray[np.floating]:
    return series.to_numpy()

In [ ]:
arr = to_price_array(df_ts["price"])

# Where We Are Heading Next


Chapter 7 uses these data structures as the state inside reusable classes.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">
